
![](https://mcd.unison.mx/wp-content/themes/awaken/img/logo_mcd.png)

# Proyecto: *Celebrity Look‑alike* con FaceNet (facenet‑pytorch)

**Objetivo:** Dada una foto (tu rostro), devolver el **Top‑k** de celebridades del dataset que más se parecen, usando **embeddings** faciales (512‑D) de un modelo preentrenado.

**Pipeline:**
1) Detección + alineación: `MTCNN` (facenet-pytorch)  
2) Embeddings: `InceptionResnetV1(pretrained="vggface2")`  
3) Clasificación ligera (fine‑tuning): una capa `Linear(512 → N)` **o** recuperación por similitud (cosine) con centroides.

> Nota: Los modelos preentrenados de facenet‑pytorch fueron entrenados con caras de **160×160**; rinden mejor si primero recortas la cara con MTCNN.


## Setup

In [1]:
!pip -q install -U --no-cache-dir scikit-learn tqdm matplotlib
!pip -q install -U --no-cache-dir --no-deps facenet-pytorch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 160.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 389.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 36.7 MB/s eta 0:00:00


In [2]:
#import torch
#print("torch:", torch.__version__)
#print("cuda available:", torch.cuda.is_available())
#print("torch cuda:", torch.version.cuda)

In [3]:
import os
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

from torchvision import datasets

from facenet_pytorch import MTCNN, InceptionResnetV1

# Reproducibilidad
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda


## Configuración

In [4]:
# Se hace la carpeta en Drive para guardar los datos
!mkdir -p /content/data

In [5]:
# Se colocan en las carpetas correspondientes
!unzip -q "/content/Open_Famous_People_Faces.zip" -d /content/data
!ls -lah /content/data

total 20K
drwxr-xr-x   3 root root 4.0K Mar 18 22:53  .
drwxr-xr-x   1 root root 4.0K Mar 18 22:53  ..
drwxrwxrwx 260 root root  12K Feb 28 18:54 'Open Famous People Faces'


In [6]:

class Config:
    # Ruta donde descomprimÍ el dataset (carpetas por celebridad)
    data_dir = "./data/Open Famous People Faces"

    # Salidas/artefactos
    out_dir = "./runs/facenet_openfamous"

    # Splits
    test_size = 0.15
    val_size = 0.15   # fracción del total (se aplica sobre el resto tras separar test)
    seed = 42

    # Embeddings
    image_size = 160
    mtcnn_margin = 20

    # Training (head)
    batch_size = 256
    epochs = 15
    lr = 1e-3
    weight_decay = 1e-4
    num_workers = 2

    # Eval
    topk = 5
    unknown_threshold = 0.35  # umbral base (lo ajustas con validación)

cfg = Config()
Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)
print(cfg)


## Cargar el dataset (ImageFolder) y crear splits estratificados

In [7]:
# Cargamos el dataset como (path, class_id) usando torchvision.datasets.ImageFolder
root = Path(cfg.data_dir)
assert root.exists(), f"No existe cfg.data_dir: {root.resolve()}"

imgfolder = datasets.ImageFolder(root=root)
samples = imgfolder.samples  # lista de (path, class_id)
class_names = imgfolder.classes

print("Clases:", len(class_names))
print("Ejemplo de clase:", class_names[24])
print("Total de imágenes:", len(samples))

Clases: 258
Ejemplo de clase: brad_pitt
Total de imágenes: 3026


In [8]:
print(samples)

[('data/Open Famous People Faces/aaron_taylor_johnson/face_detected_01ae6051.jpg', 0), ('data/Open Famous People Faces/aaron_taylor_johnson/face_detected_19597470.jpg', 0), ('data/Open Famous People Faces/aaron_taylor_johnson/face_detected_3bdf9f44.jpg', 0), ('data/Open Famous People Faces/aaron_taylor_johnson/face_detected_4015533a.jpg', 0), ('data/Open Famous People Faces/aaron_taylor_johnson/face_detected_40396e81.jpg', 0), ('data/Open Famous People Faces/aaron_taylor_johnson/face_detected_40b0e022.jpg', 0), ('data/Open Famous People Faces/aaron_taylor_johnson/face_detected_4b2d66f9.jpg', 0), ('data/Open Famous People Faces/aaron_taylor_johnson/face_detected_4bbd2080.jpg', 0), ('data/Open Famous People Faces/aaron_taylor_johnson/face_detected_58db0e97.jpg', 0), ('data/Open Famous People Faces/aaron_taylor_johnson/face_detected_5991e2bd.jpg', 0), ('data/Open Famous People Faces/aaron_taylor_johnson/face_detected_5c7cb255.jpg', 0), ('data/Open Famous People Faces/aaron_taylor_johnson/

In [9]:
print(class_names)

['aaron_taylor_johnson', 'abigail_breslin', 'adam_sandler', 'adrianne_palicki', 'alan_arkin', 'alec_baldwin', 'alexis_thorpe', 'amanda_seyfried', 'amy_adams', 'andrew_garfield', 'angelina_jolie', 'anjelica_huston', 'anna_kendrick', 'anna_paquin', 'annasophia_robb', 'anthony_hopkins', 'barbra_streisand', 'ben_affleck', 'ben_kingsley', 'ben_stiller', 'benedict_cumberbatch', 'bette_midler', 'betty_white', 'bill_murray', 'brad_pitt', 'bradley_cooper', 'brenda_fricker', 'bruce_willis', 'bryan_cranston', 'buster_keaton', 'cameron_diaz', 'carey_mulligan', 'carol_burnett', 'cary_grant', 'cate_blanchett', 'catherine_zeta_jones', 'channing_tatum', 'charlie_hunnam', 'charlize_theron', 'cher', 'chloe_grace_moretz', 'chris_cooper', 'chris_evans', 'chris_hemsworth', 'christian_bale', 'christina_ricci', 'christopher_lee', 'christopher_plummer', 'christopher_walken', 'cloris_leachman', 'colin_farrell', 'cuba_gooding_jr', 'dakota_fanning', 'daniel_craig', 'daniel_radcliffe', 'daryl_hannah', 'dave_franc

In [10]:
# Convertimos a arrays para split estratificado
paths = np.array([p for p, y in samples])
y = np.array([y for p, y in samples])

# Split: primero test, luego train/val
trainval_paths, test_paths, trainval_y, test_y = train_test_split(
    paths, y,
    test_size=cfg.test_size,
    random_state=cfg.seed,
    stratify=y
)

# val_size está definida como fracción del TOTAL; la convertimos a fracción del trainval
val_frac_of_trainval = cfg.val_size / (1.0 - cfg.test_size)

train_paths, val_paths, train_y, val_y = train_test_split(
    trainval_paths, trainval_y,
    test_size=val_frac_of_trainval,
    random_state=cfg.seed,
    stratify=trainval_y
)

print("SPLIT SIZES")
print("  train:", len(train_paths))
print("  val:  ", len(val_paths))
print("  test: ", len(test_paths))


SPLIT SIZES
  train: 2118
  val:   454
  test:  454


In [11]:
# Guardamos los splits para reproducibilidad
np.savez(
    Path(cfg.out_dir) / "splits.npz",
    train_paths=train_paths, train_y=train_y,
    val_paths=val_paths, val_y=val_y,
    test_paths=test_paths, test_y=test_y,
    class_names=np.array(class_names, dtype=object)
)
print("Splits guardados en:", Path(cfg.out_dir, "splits.npz").resolve())


Splits guardados en: /content/runs/facenet_openfamous/splits.npz


## Extractor: MTCNN (cara) + FaceNet (embeddings 512‑D)

In [12]:

# MTCNN recorta y alinea, y por defecto aplica postprocesado (prewhitening/normalización)
mtcnn = MTCNN(
    image_size=cfg.image_size,
    margin=cfg.mtcnn_margin,
    keep_all=False,   # para dataset, asumimos una cara por imagen
    device=device
)

# Modelo de embeddings (512-D por defecto)
facenet = InceptionResnetV1(pretrained="vggface2").eval().to(device)

# Nota: facenet-pytorch recomienda 160x160 y usar MTCNN antes del embedding.


  0%|          | 0.00/107M [00:00<?, ?B/s]

## Precomputar embeddings y guardarlos (.pt)

Esto acelera muchísimo el entrenamiento: luego entrenas solo una cabeza sobre vectores 512‑D.

In [13]:
@torch.no_grad()
def extraer_embeddings(paths, y, split_name):
    embs = []
    ys = []
    kept_paths = []
    skipped = 0

    for p, label in tqdm(list(zip(paths, y)), desc=f"Embeddings {split_name}", total=len(paths)):
        try:
            img = Image.open(p).convert("RGB")
        except Exception:
            skipped += 1
            continue

        face = mtcnn(img)  # Tensor [3,160,160] o None
        if face is None:
            skipped += 1
            continue

        emb = facenet(face.unsqueeze(0).to(device))  # [1,512]
        embs.append(emb.squeeze(0).cpu())
        ys.append(int(label))
        kept_paths.append(str(p))

    embs = torch.stack(embs) if len(embs) else torch.empty((0, 512))
    ys = torch.tensor(ys, dtype=torch.long)

    out = {
        "emb": embs,             # [N,512]
        "y": ys,                 # [N]
        "paths": kept_paths,     # lista[str]
        "class_names": class_names
    }
    out_path = Path(cfg.out_dir) / f"embeddings_{split_name}.pt"
    torch.save(out, out_path)
    print(f"[{split_name}] guardado: {out_path} | N={len(ys)} | skipped={skipped}")
    return out_path

In [14]:
train_pt = extraer_embeddings(train_paths, train_y, "train")
val_pt   = extraer_embeddings(val_paths,   val_y,   "val")
test_pt  = extraer_embeddings(test_paths,  test_y,  "test")


Embeddings train:   0%|          | 0/2118 [00:00<?, ?it/s]

[train] guardado: runs/facenet_openfamous/embeddings_train.pt | N=1055 | skipped=1063


Embeddings val:   0%|          | 0/454 [00:00<?, ?it/s]

[val] guardado: runs/facenet_openfamous/embeddings_val.pt | N=223 | skipped=231


Embeddings test:   0%|          | 0/454 [00:00<?, ?it/s]

[test] guardado: runs/facenet_openfamous/embeddings_test.pt | N=230 | skipped=224


In [17]:
print(train_pt)
print(val_pt)
print(test_pt)

runs/facenet_openfamous/embeddings_train.pt
runs/facenet_openfamous/embeddings_val.pt
runs/facenet_openfamous/embeddings_test.pt


## DataLoaders de embeddings (Dataset sencillo)

In [18]:

class EmbeddingsDataset(Dataset):
    def __init__(self, pt_path: str):
        d = torch.load(pt_path, map_location="cpu")
        self.emb = d["emb"].float()
        self.y = d["y"].long()
        self.paths = d["paths"]
        self.class_names = list(d["class_names"])

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.emb[idx], self.y[idx]

def crear_dataloaders(cfg, train_pt, val_pt, test_pt):
    train_ds = EmbeddingsDataset(train_pt)
    val_ds = EmbeddingsDataset(val_pt)
    test_ds = EmbeddingsDataset(test_pt)

    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True
    )
    test_loader = DataLoader(
        test_ds, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True
    )
    return train_loader, val_loader, test_loader, train_ds.class_names

In [19]:

train_loader, val_loader, test_loader, class_names = crear_dataloaders(cfg, train_pt, val_pt, test_pt)
num_classes = len(class_names)
print("num_classes:", num_classes)


num_classes: 258


## Modelo: Head de clasificación sobre embeddings (fine‑tuning ligero)

In [20]:

class EmbeddingClassifier(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.fc = nn.Linear(512, num_classes)

    def forward(self, emb):
        return self.fc(emb)

model = EmbeddingClassifier(num_classes).to(device)

# Solo entrenamos el head
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
criterion = nn.CrossEntropyLoss()

print(model)


EmbeddingClassifier(
  (fc): Linear(in_features=512, out_features=258, bias=True)
)


## Funciones de métrica y entrenamiento por epoch

In [21]:

def topk_accuracy(logits, y, k: int = 5) -> torch.Tensor:
    # logits: [B,C], y: [B]
    k = min(k, logits.size(1))
    topk = torch.topk(logits, k=k, dim=1).indices  # [B,k]
    correct = (topk == y.unsqueeze(1)).any(dim=1).float()  # [B]
    return correct.mean()

def ejecutar_epoca(model, dataloader, device, optimizer=None):
    entrenando = optimizer is not None
    model.train(entrenando)

    total_loss = 0.0
    total_top1 = 0.0
    total_topk = 0.0
    total_batches = 0

    for emb, y in tqdm(dataloader, desc="train" if entrenando else "eval", leave=False):
        emb = emb.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if entrenando:
            optimizer.zero_grad()

        with torch.set_grad_enabled(entrenando):
            logits = model(emb)
            loss = criterion(logits, y)

            if entrenando:
                loss.backward()
                optimizer.step()

        with torch.no_grad():
            top1 = (logits.argmax(dim=1) == y).float().mean()
            topk = topk_accuracy(logits, y, k=cfg.topk)

        total_loss += loss.item()
        total_top1 += top1.item()
        total_topk += topk.item()
        total_batches += 1

    return {
        "loss": total_loss / total_batches,
        "top1": total_top1 / total_batches,
        f"top{cfg.topk}": total_topk / total_batches,
    }


## Entrenamiento



In [22]:

best_val = -1.0
best_path = Path(cfg.out_dir) / "best_head.pt"

for epoch in range(1, cfg.epochs + 1):
    print(f"\nEpoch {epoch}/{cfg.epochs}")
    train_metrics = ejecutar_epoca(model, train_loader, device, optimizer)
    val_metrics = ejecutar_epoca(model, val_loader, device, None)

    print(
        f'train_loss={train_metrics["loss"]:.4f} | train_top1={train_metrics["top1"]:.4f} | train_top{cfg.topk}={train_metrics[f"top{cfg.topk}"]:.4f}\n'
        f'val_loss={val_metrics["loss"]:.4f}   | val_top1={val_metrics["top1"]:.4f}   | val_top{cfg.topk}={val_metrics[f"top{cfg.topk}"]:.4f}'
    )

    if val_metrics["top1"] > best_val:
        best_val = val_metrics["top1"]
        torch.save({"model": model.state_dict(), "class_names": class_names, "cfg": cfg.__dict__}, best_path)
        print("  ✅ Nuevo mejor modelo guardado:", best_path)

print("\nBest val top1:", best_val)



Epoch 1/15


train:   0%|          | 0/5 [00:00<?, ?it/s]

eval:   0%|          | 0/1 [00:00<?, ?it/s]

train_loss=5.5343 | train_top1=0.0334 | train_top5=0.0877
val_loss=5.4967   | val_top1=0.1256   | val_top5=0.3229
  ✅ Nuevo mejor modelo guardado: runs/facenet_openfamous/best_head.pt

Epoch 2/15


train:   0%|          | 0/5 [00:00<?, ?it/s]

...

eval:   0%|          | 0/1 [00:00<?, ?it/s]

train_loss=5.4160 | train_top1=0.5630 | train_top5=0.7963
val_loss=5.3964   | val_top1=0.6592   | val_top5=0.8072
  ✅ Nuevo mejor modelo guardado: runs/facenet_openfamous/best_head.pt

Epoch 4/15


train:   0%|          | 0/5 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b0fb4577d80><function _MultiProcessingDataLoaderIter.__del__ at 0x7b0fb4577d80>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()    
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
self._shutdown_workers()
    if w.is_alive():  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

      if w.is_alive(): 
       ^^ ^ Exception ignored in: ^^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b0fb4577d80> ^^^
^Traceback (most recent call last):
^^^Exception ignored in:   File "/usr/local/lib/python3.12/dist-packag

RuntimeError: DataLoader worker (pid(s) 24741) exited unexpectedly